# Sentence Embeddings — Day 1 Notebook

> **Goal:** Go from raw text → tokens → TF-IDF vectors → cosine similarity → 2D visualisation.  
> Then explore Word2Vec embeddings with gensim.

## What you will build
```
Raw text
   ↓  tokenize + clean
Tokens
   ↓  TF-IDF vectorizer
Sparse vectors (N × vocab_size)
   ↓  cosine similarity
Similarity matrix
   ↓  PCA
2D scatter plot (clusters = semantic groups)
```

## 0. Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import nltk
for resource in ['punkt', 'punkt_tab', 'stopwords', 'wordnet']:
    nltk.download(resource, quiet=True)

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import PCA

sns.set_theme(style='whitegrid', palette='husl')
print('✓ All imports OK')

## 1. Corpus — 20 sentences about AI/ML

In [ ]:
CORPUS = [
    # --- Machine Learning ---
    'Machine learning enables computers to learn from data.',
    'Supervised learning uses labelled training examples.',
    'Unsupervised learning finds hidden patterns without labels.',
    'Reinforcement learning trains agents through reward signals.',
    'Cross-validation helps estimate true model performance.',
    
    # --- Deep Learning ---
    'Deep learning uses neural networks with many layers.',
    'Convolutional networks excel at image recognition tasks.',
    'Recurrent networks model sequential and temporal data.',
    'Transformers revolutionised natural language processing in 2017.',
    'BERT and GPT are large pre-trained language models.',
    
    # --- NLP ---
    'Tokenisation splits text into individual word units.',
    'Stop word removal filters out common high-frequency words.',
    'Lemmatisation converts words to their base dictionary form.',
    'Named entity recognition identifies people places and organisations.',
    'Sentiment analysis determines if text is positive or negative.',
    
    # --- Embeddings ---
    'Word embeddings represent words as dense numerical vectors.',
    'Cosine similarity measures the angle between two vectors.',
    'Word2Vec learns embeddings by predicting surrounding words.',
    'GloVe embeddings use global word co-occurrence statistics.',
    'Sentence embeddings encode entire sentences into single vectors.',
]

# Group labels for colouring the scatter plot
GROUP_LABELS = (
    ['ML'] * 5 +
    ['Deep Learning'] * 5 +
    ['NLP'] * 5 +
    ['Embeddings'] * 5
)

print(f'Corpus: {len(CORPUS)} sentences across {len(set(GROUP_LABELS))} topics')
pd.DataFrame({'sentence': CORPUS, 'group': GROUP_LABELS})

## 2. Tokenisation & Preprocessing Pipeline

In [ ]:
import string

STOP_WORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text: str) -> list[str]:
    tokens = word_tokenize(text.lower())
    tokens = [t for t in tokens if t not in string.punctuation]
    tokens = [t for t in tokens if t not in STOP_WORDS]
    tokens = [lemmatizer.lemmatize(t) for t in tokens]
    return tokens

# Demo on first sentence
example = CORPUS[0]
print(f'Original  : {example}')
print(f'Processed : {preprocess(example)}')

# Apply to full corpus
processed = [preprocess(s) for s in CORPUS]
print(f'\nProcessed all {len(processed)} sentences.')

## 3. TF-IDF Vectors

$$\text{TF-IDF}(t,d) = \underbrace{\frac{f_{t,d}}{\sum_k f_{k,d}}}_{\text{TF}} \times \underbrace{\log\frac{N}{df_t}}_{\text{IDF}}$$

- **TF** — how often the word appears in *this* document  
- **IDF** — penalises words that appear in *many* documents ("the", "is")  
- Result: rare-but-important words get high scores

In [ ]:
vectorizer = TfidfVectorizer(stop_words='english', max_features=50)
tfidf_matrix = vectorizer.fit_transform(CORPUS)   # sparse (20 × 50)
vocab = vectorizer.get_feature_names_out()

print(f'TF-IDF matrix shape: {tfidf_matrix.shape}  (docs × vocab)')
print(f'Vocabulary ({len(vocab)} terms): {list(vocab[:20])}')

# Show top-5 terms for first 3 sentences
dense = tfidf_matrix.toarray()
for i in range(3):
    top_idx = dense[i].argsort()[::-1][:5]
    top_terms = [(vocab[j], round(dense[i, j], 3)) for j in top_idx if dense[i, j] > 0]
    print(f'\nDoc {i}: "{CORPUS[i][:55]}..."')
    print(f'  Top TF-IDF terms: {top_terms}')

## 4. Cosine Similarity Matrix

$$\cos(\mathbf{A}, \mathbf{B}) = \frac{\mathbf{A} \cdot \mathbf{B}}{\|\mathbf{A}\| \|\mathbf{B}\|}$$

- **1.0** = identical direction (same topic)
- **0.0** = perpendicular (unrelated topics)
- We expect ML/DL sentences to be similar to each other and different from NLP/Embeddings.

In [ ]:
sim_matrix = cosine_similarity(tfidf_matrix)   # (20 × 20)

fig, ax = plt.subplots(figsize=(11, 9))

short_labels = [f'{g[:2]}{i%5}' for i, g in enumerate(GROUP_LABELS)]
sns.heatmap(
    sim_matrix, annot=False, cmap='YlOrRd',
    xticklabels=short_labels, yticklabels=short_labels,
    vmin=0, vmax=1, linewidths=0.3, ax=ax
)

# Draw group boundaries
for boundary in [5, 10, 15]:
    ax.axhline(boundary, color='navy', linewidth=1.5)
    ax.axvline(boundary, color='navy', linewidth=1.5)

ax.set_title('Cosine Similarity Matrix\n(block-diagonal = within-topic similarity)',
             fontsize=13, fontweight='bold')

# Add group labels on the side
groups = ['ML', 'Deep\nLearning', 'NLP', 'Embeddings']
for i, g in enumerate(groups):
    ax.text(-0.5, (i * 5) + 2.5, g, ha='right', va='center',
            fontsize=8, color='navy', fontweight='bold',
            transform=ax.get_yaxis_transform())

plt.tight_layout()
plt.show()

# Show most similar pair (excluding self)
np.fill_diagonal(sim_matrix, 0)
i, j = np.unravel_index(sim_matrix.argmax(), sim_matrix.shape)
print(f'Most similar pair (score={sim_matrix[i,j]:.3f}):')
print(f'  [{i}] {CORPUS[i]}')
print(f'  [{j}] {CORPUS[j]}')

## 5. Semantic Search

Given a query, find the most relevant sentences using cosine similarity.

In [ ]:
def semantic_search(query: str, top_n: int = 3) -> pd.DataFrame:
    """Embed query with the same TF-IDF vectorizer, rank by cosine similarity."""
    query_vec = vectorizer.transform([query])          # same vocabulary!
    scores = cosine_similarity(query_vec, tfidf_matrix)[0]
    
    ranked_idx = scores.argsort()[::-1][:top_n]
    return pd.DataFrame({
        'rank':  range(1, top_n + 1),
        'score': [round(scores[i], 4) for i in ranked_idx],
        'group': [GROUP_LABELS[i] for i in ranked_idx],
        'sentence': [CORPUS[i] for i in ranked_idx],
    })

# Test 3 different queries
for query in [
    'how do neural networks work',
    'word vector representations',
    'training data labels classification',
]:
    print(f'\nQuery: "{query}"')
    display(semantic_search(query, top_n=3))

## 6. PCA — Project to 2D

PCA finds the axes of maximum variance in high-dimensional data and projects onto 2 dimensions.  
Similar sentences (same topic) should cluster together.

In [ ]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(dense)   # (20, 50) → (20, 2)

explained = pca.explained_variance_ratio_
print(f'PC1 explains {explained[0]:.1%} of variance')
print(f'PC2 explains {explained[1]:.1%} of variance')
print(f'Total: {explained.sum():.1%}')

# Visualise
palette = {'ML': '#e74c3c', 'Deep Learning': '#3498db',
           'NLP': '#2ecc71', 'Embeddings': '#9b59b6'}

fig, ax = plt.subplots(figsize=(11, 8))

for group in set(GROUP_LABELS):
    idx = [i for i, g in enumerate(GROUP_LABELS) if g == group]
    ax.scatter(
        coords[idx, 0], coords[idx, 1],
        label=group, color=palette[group],
        s=140, alpha=0.85, edgecolors='white', linewidths=0.8
    )

for i, sentence in enumerate(CORPUS):
    # Show only first 4 words
    short = ' '.join(sentence.split()[:4])
    ax.annotate(short, (coords[i, 0], coords[i, 1]),
                fontsize=7, xytext=(4, 3), textcoords='offset points', alpha=0.85)

ax.axhline(0, color='lightgray', linewidth=0.8, linestyle='--')
ax.axvline(0, color='lightgray', linewidth=0.8, linestyle='--')
ax.set_xlabel(f'PC1 ({explained[0]:.1%} variance)', fontsize=11)
ax.set_ylabel(f'PC2 ({explained[1]:.1%} variance)', fontsize=11)
ax.set_title('Sentence Embeddings in 2D\n(TF-IDF → PCA)', fontsize=13, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

## 7. Word2Vec — Dense Embeddings

TF-IDF creates **sparse** vectors (mostly zeros). Word2Vec creates **dense** vectors where semantically similar words are close.

```
TF-IDF:   'learning' = [0, 0, 1, 0, 0, 0, ..., 0]   # 50 dims, mostly 0
Word2Vec: 'learning' = [0.23, -0.45, 0.78, ...]       # 50 dims, all non-zero
```

In [ ]:
try:
    from gensim.models import Word2Vec
    GENSIM_OK = True
except ImportError:
    GENSIM_OK = False
    print('gensim not installed. Run: uv add gensim')

if GENSIM_OK:
    tokenized = [preprocess(s) for s in CORPUS]
    
    model = Word2Vec(
        sentences=tokenized,
        vector_size=50,
        window=3,
        min_count=1,
        sg=1,        # skip-gram
        epochs=500,
        seed=42,
    )
    
    wv = model.wv
    print(f'Vocabulary size: {len(wv)}')
    print(f"\n'learning' vector (first 10 dims): {wv['learning'][:10].round(3)}")
    
    print('\n--- Nearest neighbours ---')
    for word in ['learning', 'neural', 'word', 'vector']:
        if word in wv:
            neighbours = wv.most_similar(word, topn=3)
            print(f"  '{word}' → {[(w, round(s, 3)) for w, s in neighbours]}")

## 8. Word2Vec — 2D Visualisation

In [ ]:
if GENSIM_OK and len(wv) >= 10:
    # Pick interesting words
    words_to_plot = [
        'learning', 'machine', 'deep', 'neural', 'network',
        'language', 'text', 'word', 'vector', 'embedding',
        'python', 'data', 'model', 'train', 'feature'
    ]
    words_to_plot = [w for w in words_to_plot if w in wv]
    
    word_vecs = np.array([wv[w] for w in words_to_plot])
    
    pca2 = PCA(n_components=2, random_state=42)
    w2v_2d = pca2.fit_transform(word_vecs)
    
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.scatter(w2v_2d[:, 0], w2v_2d[:, 1],
               s=120, c=range(len(words_to_plot)), cmap='tab20',
               alpha=0.85, edgecolors='white', linewidths=0.8)
    
    for i, word in enumerate(words_to_plot):
        ax.annotate(word, (w2v_2d[i, 0], w2v_2d[i, 1]),
                    fontsize=10, fontweight='bold',
                    xytext=(5, 4), textcoords='offset points')
    
    ax.axhline(0, color='lightgray', linewidth=0.7, linestyle='--')
    ax.axvline(0, color='lightgray', linewidth=0.7, linestyle='--')
    ax.set_title('Word2Vec Embeddings in 2D (PCA)\nSemantically similar words cluster together',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('PC1', fontsize=11)
    ax.set_ylabel('PC2', fontsize=11)
    
    plt.tight_layout()
    plt.show()

## 9. Summary — BoW vs TF-IDF vs Word2Vec

| | BoW | TF-IDF | Word2Vec |
|---|---|---|---|
| Vector type | Sparse | Sparse | **Dense** |
| Captures frequency | ✓ | ✓ (weighted) | ✗ |
| Captures word importance | ✗ | ✓ | ✗ |
| Captures semantics | ✗ | ✗ | **✓** |
| `king - man + woman ≈ queen` | ✗ | ✗ | **✓** |
| Training data needed | ✗ | ✗ | Large corpus |
| Best for | Baseline search | Search / ranking | Semantic tasks, NLU |

**Modern production**: use transformer-based sentence encoders (Sentence-BERT, OpenAI embeddings) which produce 768–1536 dim dense vectors capturing full sentence context.

In [ ]:
# Final comparison: TF-IDF vs W2V average embedding similarity
if GENSIM_OK:
    def sentence_to_w2v(tokens: list[str]) -> np.ndarray:
        """Average Word2Vec vectors for all known words."""
        vecs = [wv[t] for t in tokens if t in wv]
        return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

    query = 'word vector representation'
    query_tokens = preprocess(query)

    results = []
    for i, (sentence, tokens) in enumerate(zip(CORPUS, tokenized)):
        tfidf_score = cosine_similarity(
            vectorizer.transform([query]),
            vectorizer.transform([sentence])
        )[0, 0]
        w2v_score = cosine_similarity(
            sentence_to_w2v(query_tokens).reshape(1, -1),
            sentence_to_w2v(tokens).reshape(1, -1)
        )[0, 0]
        results.append({'sentence': sentence[:60], 'tfidf': round(tfidf_score, 4),
                        'word2vec': round(w2v_score, 4)})

    df_compare = pd.DataFrame(results).sort_values('tfidf', ascending=False).head(8)
    print(f'Query: "{query}"')
    display(df_compare)